# A01: Concurrencia y Paralelismo - Fundamentos

<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); color: white; padding: 30px; border-radius: 12px; margin-bottom: 20px;">
<h2 style="margin:0; font-size:1.4em;">Advanced Python Series</h2>
<p style="margin:8px 0 0 0; opacity:0.85;">Módulo 01 de 06 | Duración estimada: 90 min</p>
</div>

En este notebook exploraremos los fundamentos teóricos y prácticos de la **concurrencia** y el **paralelismo** en Python. Aprenderemos a identificar qué tipo de trabajo realizan nuestros programas, cómo el GIL afecta el rendimiento, y cuándo usar hilos, procesos o asincronía.

---
## Objetivos de Aprendizaje

Al finalizar este notebook serás capaz de:

1. **Distinguir** entre concurrencia (interleaving) y paralelismo (simultáneo) con precisión técnica.
2. **Clasificar** cualquier tarea como I/O-bound o CPU-bound y elegir la herramienta correcta.
3. **Explicar** qué es el GIL, por qué existe, y cómo afecta el rendimiento de hilos en Python.
4. **Implementar** soluciones con `threading`, `multiprocessing` y `asyncio`, midiendo su rendimiento real.
5. **Aplicar** técnicas de benchmarking para comparar estrategias de concurrencia de forma rigurosa.

---
## Analogía Central

Imagina una **cocina profesional**:

| Concepto | Analogía |
|----------|----------|
| **I/O Bound** | Esperar a que llegue un paquete de ingredientes. No estás cocinando, solo *esperas*. Puedes atender otra tarea mientras tanto. |
| **CPU Bound** | Cocinar en un horno de una sola entrada. Si necesitas hornear 10 pasteles, uno debe terminar antes de empezar el siguiente. |
| **GIL (Global Interpreter Lock)** | Una sola **llave de acceso** a la cocina. Solo un cocinero puede usar la llave a la vez. Si ambos necesitan el horno (recurso compartido), deben turnarse. |
| **Paralelismo** | Tienes **10 cocinas idénticas**, cada una con su horno. Puedes hornear 10 pasteles *simultáneamente*. |
| **Concurrencia** | Un solo cocinero que **alterna** entre tareas: prepara masa, mientras se hornea el pastel, lava trastes. Nunca hace dos cosas a la vez, pero avanza en varias. |

> **Clave**: La concurrencia es sobre *gestionar* múltiples tareas. El paralelismo es sobre *ejecutar* múltiples tareas simultáneamente.

---
## 1. Concurrencia vs Paralelismo

Aunque se usan como sinónimos, son conceptos **fundamentalmente diferentes**.

### Definiciones

| | Concurrencia | Paralelismo |
|---|---|---|
**Definición** | Múltiples tareas *progresan* en un periodo de tiempo, alternando ejecución | Múltiples tareas se ejecutan *al mismo tiempo* en distintos cores/procesadores |
**Metáfora** | Un cocinero alternando entre preparar ensalada y vigilar la sopa | Dos cocineros trabajando en recetas distintas simultáneamente |
**Recurso** | Puede ejecutarse en **1 core** | Requiere **múltiples cores** |
**Herramienta** | threading, asyncio | multiprocessing |
**Cuándo** | I/O-bound (esperas de red/disco) | CPU-bound (cálculos pesados) |

### Diagrama: Concurrencia (1 core)

```
Core Único:
┌──────────────────────────────────────────────────────┐
│  Hilo A    ████████░░░░░░░░████████░░░░░░████████░░  │
│  Hilo B    ░░░░░░░░████████░░░░░░░░████████░░░░░░░░  │
│  Hilo C    ░░░░████████░░░░░░░░░░░░░░░░░░░░████████  │
└──────────────────────────────────────────────────────┘
           ████ = ejecutándose   ░░░░ = esperando (GIL liberado)

→ Solo UNO ejecuta a la vez. El scheduler alterna entre hilos.
→ Gana eficiencia cuando los hilos están en I/O (no consumen CPU).
```

### Diagrama: Paralelismo (N cores)

```
Core 1: ████████████████████████████████  ← Proceso A
Core 2: ████████████████████████████████  ← Proceso B
Core 3: ████████████████████████████████  ← Proceso C
Core 4: ████████████████████████████████  ← Proceso D

→ Cada core ejecuta un proceso de forma SIMULTÁNEA.
→ No hay contention: cada proceso tiene su propio intérprete y memoria.
```

### Diagrama: Paralelismo Estructural (asyncio)

```
Event Loop (único hilo):
┌────────────────────────────────────────────────────────────┐
│  await fetch_url_1()  →  ↑ callback → await fetch_url_2() │
│         │                                              ↑  │
│         ↓ (cede control, no bloquea el loop)            │  │
└────────────────────────────────────────────────────────────┘

→ UN solo hilo, pero gestiona miles de conexiones.
→ Ideal para I/O masivo (servidores web, scrapers).
```

### ¿Cuándo importa?

- **Concurrencia importa** cuando tu programa pasa tiempo *esperando* (red, disco, base de datos). Puedes aprovechar esas esperas para hacer otra cosa.
- **Paralelismo importa** cuando tu programa pasa tiempo *calculando*. Necesitas más cores para ejecutar más rápido.
- **Ambos importan** en aplicaciones reales: un servidor web recibe peticiones (I/O) que requieren procesamiento (CPU).

---
## 2. Tipos de trabajo: I/O Bound vs CPU Bound

El **primer paso** para escribir código concurrente eficiente es identificar qué tipo de trabajo realiza tu programa.

### I/O Bound (Bound by Input/Output)

El programa pasa la mayor parte del tiempo **esperando** una operación externa:

| Fuente de espera | Ejemplo |
|---|---|
| Red | `requests.get()`, `urllib`, llamadas a APIs |
| Disco | `open()`, `read()`, `write()` en archivos grandes |
| Base de datos | `SELECT` con latencia, `cursor.fetchall()` |
| Consola | `input()`, lectura de stdin |

**Señales de I/O bound:**
- El CPU está *ocioso* la mayor parte del tiempo.
- `htop` muestra CPU bajo pero el proceso sigue activo.
- Agregar más cores no acelera el programa.

### CPU Bound (Bound by CPU)

El programa pasa la mayor parte del tiempo **calculando**:

| Fuente de carga | Ejemplo |
|---|---|
| Cálculos matemáticos | Suma de primos, Fibonacci recursivo, cálculo de PI |
| Procesamiento de datos | Transformaciones pesadas con pandas, loops anidados |
| Compresión | `gzip.compress()`, `zlib.compress()` |
| Criptografía | `hashlib`, `bcrypt` |
| Machine Learning | Entrenamiento sin GPU, validación cruzada |

**Señales de CPU bound:**
- `htop` muestra un core al 100%.
- El tiempo de ejecución es predecible (no depende de red/disco).
- Agregar más cores *sí* aceleraría.

### Método Heurístico: ¿Qué tipo de trabajo soy?

```
¿Tu programa hace llamadas a red, disco o DB?
  ├─ SÍ → ¿La mayor parte del tiempo es I/O?
  │         ├─ SÍ → I/O Bound
  │         └─ NO → ¿Mezcla de ambos?
  │                   ├─ SÍ → Identifica la BOTLENECK principal
  │                   └─ (tratar como I/O para concurrencia, CPU para paralelismo)
  └─ NO → ¿El tiempo es proporcional al tamaño de entrada y cálculos?
            ├─ SÍ → CPU Bound
            └─ NO → Revisa tu algoritmo primero
```

In [ ]:
import time

# Demo: identificar el tipo de trabajo midiendo tiempo de CPU vs tiempo real

def ejemplo_io_bound(n: int = 5) -> None:
    """Simula I/O: time.sleep simula esperar respuesta de red/disco."""
    for i in range(n):
        time.sleep(0.1)  # 100ms de "espera"
    print(f"I/O bound completado: {n} iteraciones")


def ejemplo_cpu_bound(n: int = 5) -> None:
    """CPU bound: cálculo intensivo puro."""
    total = 0
    for i in range(n * 5_000_000):
        total += i * i
    print(f"CPU bound completado: suma parcial = {total}")


# Medir el tipo de trabajo
print("=" * 50)
print("EJEMPLO I/O BOUND")
print("=" * 50)

inicio = time.perf_counter()
ejemplo_io_bound(5)
real_time = time.perf_counter() - inicio

print(f"Tiempo real: {real_time:.3f}s")
print(f"Tiempo CPU (aprox): ~0.001s")
print(f"Ratio CPU/Real: ~{0.001/real_time:.1%} → El CPU está OCIOSO")

print()
print("=" * 50)
print("EJEMPLO CPU BOUND")
print("=" * 50)

inicio = time.perf_counter()
ejemplo_cpu_bound(5)
real_time = time.perf_counter() - inicio

print(f"Tiempo real: {real_time:.3f}s")
print(f"Ratio CPU/Real: ~100% → El CPU está TRABAJANDO")

> **Regla práctica**: Si el tiempo real es mucho mayor que el tiempo de CPU, es I/O bound. Si son similares, es CPU bound.

---
## 3. El GIL (Global Interpreter Lock)

El GIL es el factor **más importante** que debes entender sobre concurrencia en Python.

### ¿Qué es?

El GIL es un **mutex** (mutual exclusion lock) que protege el acceso al intérprete de Python CPython. Solo **un hilo puede ejecutar bytecode de Python a la vez**, incluso en máquinas con múltiples cores.

### ¿Por qué existe?

Fue introducido por **Guido van Rossum** en 1991 por razones prácticas:

1. **Gestión de memoria**: CPython usa conteo de referencias (`refcounting`). Sin el GIL, dos hilos podrían modificar el conteo de referencias de un objeto simultáneamente → memory leaks o doble-free.
2. **Simplicidad**: Hace el intérprete más fácil de implementar y mantener.
3. **Bibliotecas C**: Muchas extensiones C asumen que el GIL está presente y no son thread-safe.

### Diagrama del GIL

```
╔═══════════════════════════════════════════════════════════════╗
║                    PYTHON (CPython)                          ║
║                                                              ║
║  ┌─────────┐  ┌─────────┐  ┌─────────┐                      ║
║  │ Hilo 1  │  │ Hilo 2  │  │ Hilo 3  │                      ║
║  │ ████░░  │  │ ░░████  │  │ ░░░░░░  │                      ║
║  └────┬────┘  └────┬────┘  └────┬────┘                      ║
║       │            │            │                            ║
║       └────────────┼────────────┘                            ║
║                    │                                         ║
║             ┌──────┴──────┐                                  ║
║             │     GIL     │  ← SOLO UNO puede pasar a la vez ║
║             │  🔒 (llave)  │                                  ║
║             └──────┬──────┘                                  ║
║                    │                                         ║
║             ┌──────┴──────┐                                  ║
║             │  Bytecode   │                                  ║
║             │  Executor   │                                  ║
║             └─────────────┘                                  ║
║                                                              ║
║  IO操作 (sleep,网络,disco): Se LIBERA el GIL temporalmente   ║
║  CPU puro: El GIL NO se libera → NO hay paralelismo real     ║
╚═══════════════════════════════════════════════════════════════╝
```

### ¿Cuándo se libera el GIL?

| Operación | ¿Se libera el GIL? |
|---|---|
| `time.sleep()` | Sí |
| Lectura/escritura de archivo | Sí |
| Operaciones de red (`socket`, `requests`) | Sí |
| Cálculos en Python puro | **No** |
| Operaciones de NumPy (C-level) | Sí (liberado internamente) |
| `subprocess.run()` | Sí (proceso separado) |

### Demo: El GIL en acción

Si los hilos ejecutan **CPU puro**, el GIL impide paralelismo real. Veámoslo:

In [ ]:
import threading
import time

def cpu_work(iterations: int) -> None:
    """Trabajo CPU puro: suma acumulada."""
    total = 0
    for i in range(iterations):
        total += i
    return total


# --- Secuencial ---
ITERATIONS = 20_000_000

print("Secuencial:")
start = time.perf_counter()
cpu_work(ITERATIONS)
cpu_work(ITERATIONS)
sequential = time.perf_counter() - start
print(f"  Tiempo: {sequential:.3f}s")

# --- Con 2 hilos (CPU bound) ---
print("Con 2 hilos (CPU bound):")
start = time.perf_counter()
t1 = threading.Thread(target=cpu_work, args=(ITERATIONS,))
t2 = threading.Thread(target=cpu_work, args=(ITERATIONS,))
t1.start()
t2.start()
t1.join()
t2.join()
threaded = time.perf_counter() - start
print(f"  Tiempo: {threaded:.3f}s")

print(f"\nRatio (hilos / secuencial): {threaded/sequential:.2f}x")
print("→ Los hilos NO aceleran CPU bound. El GIL los serializa.")
print(f"→ De hecho, {"son más lentos" if threaded > sequential else "son similares"} por el overhead del GIL.")

### El GIL y las librerías C

Algunas librerías **liberan el GIL** internamente durante operaciones pesadas:

- **NumPy**: Operaciones vectorizadas (linalg, dot products) liberan el GIL.
- **Pandas**: Internamente usa NumPy, así que muchas operaciones también lo liberan.
- **SciPy**: Similar a NumPy.

Esto significa que con librerías C bien diseñadas, `threading` *puede* acelerar ciertas operaciones numéricas — pero no es confiable.

### La solución para CPU bound: `multiprocessing`

```
Threading (GIL):              multiprocessing (sin GIL):

Core 1: ████░░████░░████       Core 1: ████████████████  ← Proceso A
Core 2: ░░████░░████░░░░       Core 2: ████████████████  ← Proceso B
         ↑ comparten GIL                ↑ cada uno tiene su propio GIL
```

---
## 4. Hilos (`threading`)

**Cuándo usar**: Tareas I/O-bound donde el tiempo de espera domina.

In [ ]:
import threading
import time

def descargar(nombre: str, segundos: float) -> str:
    """Simula descarga de un archivo (I/O bound)."""
    print(f"  [INICIO] Descargando {nombre}...")
    time.sleep(segundos)  # Simula espera de red (libera GIL)
    msg = f"  [LISTO] {nombre} descargado ({segundos}s)"
    print(msg)
    return msg


# --- Secuencial ---
print("=" * 50)
print("DESCARGAS SECUENCIALES")
print("=" * 50)

start = time.perf_counter()
descargar("archivo_A.dat", 0.5)
descargar("archivo_B.dat", 0.5)
descargar("archivo_C.dat", 0.5)
secuencial = time.perf_counter() - start
print(f"  Total secuencial: {secuencial:.3f}s")

# --- Con hilos ---
print()
print("=" * 50)
print("DESCARGAS CON THREADING")
print("=" * 50)

start = time.perf_counter()
hilos = []
for nombre, duracion in [("archivo_A.dat", 0.5), ("archivo_B.dat", 0.5), ("archivo_C.dat", 0.5)]:
    t = threading.Thread(target=descargar, args=(nombre, duracion))
    hilos.append(t)
    t.start()

for t in hilos:
    t.join()

con_hilos = time.perf_counter() - start
print(f"  Total con hilos: {con_hilos:.3f}s")
print(f"\n  Speedup: {secuencial / con_hilos:.2f}x")
print("  → En I/O bound, los hilos ACCELERAN porque liberan el GIL durante sleep/IO.")

### `Thread.start()` vs `Thread.join()`

| Método | Qué hace |
|--------|----------|
| `start()` | Inicia el hilo y comienza a ejecutar la función target **en paralelo** |
| `join()` | El hilo actual **bloquea** hasta que el hilo llamado termine |

```python
t = Thread(target=mi_funcion)
t.start()   # No bloquea: el hilo corre en background
# ... hacer otras cosas ...
t.join()    # Bloquea aquí hasta que termine el hilo
```

### `threading.Event` y `threading.Lock`

Para sincronización entre hilos:

In [ ]:
import threading
import time

# Ejemplo con Lock: protección de recurso compartido
contador = 0
lock = threading.Lock()

def incrementar_seguro(n: int) -> None:
    global contador
    for _ in range(n):
        with lock:  # Solo un hilo a la vez puede modificar
            contador += 1


def incrementar_inseguro(n: int) -> None:
    global contador
    for _ in range(n):
        contador += 1  # ¡Race condition!


# Demo: incrementar sin lock (race condition)
contador = 0
hilos = [threading.Thread(target=incrementar_inseguro, args=(100_000,)) for _ in range(10)]
for t in hilos:
    t.start()
for t in hilos:
    t.join()
print(f"Sin lock  (esperado 1,000,000): {contador:,}")

# Demo: incrementar con lock (seguro)
contador = 0
hilos = [threading.Thread(target=incrementar_seguro, args=(100_000,)) for _ in range(10)]
for t in hilos:
    t.start()
for t in hilos:
    t.join()
print(f"Con lock  (esperado 1,000,000): {contador:,}")

---
## 5. Procesos (`multiprocessing`)

**Cuándo usar**: Tareas CPU-bound donde necesitas paralelismo real.

In [ ]:
from multiprocessing import Process, Pool
import time


def cpu_heavy(n: int) -> int:
    """Cálculo CPU intensivo."""
    total = 0
    for i in range(n):
        total += i * i
    return total


def worker(args: tuple[int, int]) -> int:
    """Worker para Pool."""
    start, end = args
    return sum(i * i for i in range(start, end))


ITERATIONS = 30_000_000

# --- Secuencial ---
print("Secuencial:")
start = time.perf_counter()
cpu_heavy(ITERATIONS)
cpu_heavy(ITERATIONS)
cpu_heavy(ITERATIONS)
cpu_heavy(ITERATIONS)
sequential = time.perf_counter() - start
print(f"  Tiempo: {sequential:.3f}s")

# --- Con Pool de procesos ---
print("\nCon Pool de 4 procesos:")
start = time.perf_counter()
chunk = ITERATIONS // 4
args_list = [(i * chunk, (i + 1) * chunk) for i in range(4)]

with Pool(processes=4) as pool:
    resultados = pool.map(worker, args_list)

parallel = time.perf_counter() - start
print(f"  Tiempo: {parallel:.3f}s")
print(f"  Speedup: {sequential / parallel:.2f}x")
print("\n→ multiprocessing SÍ acelera CPU bound porque cada proceso tiene su propio GIL.")

### `Process` vs `Pool`

| | `Process` | `Pool` |
|---|---|---|
**Uso** | Control fino: creas, inicias y joins cada proceso | Trabajo en lote: envías tareas y recibes resultados |
**API** | `p = Process(target=fn); p.start(); p.join()` | `pool.map(fn, iterable)` |
**Ideal para** | Número fijo de procesos con lógica específica | Dividir un iterable entre N workers |
**Overhead** | Menor (creas lo que necesitas) | Mayor (gestiona cola de tareas) |

---
## 6. Asincronía (`asyncio`) - Vista Previa

`asyncio` implementa concurrencia basada en **cooperativa** (no preemptiva) usando un **event loop**.

### Conceptos clave

| Concepto | Descripción |
|----------|-------------|
| **Coroutine** | Función definida con `async def`. Se puede *pausar* y *reanudar*. |
| **`await`** | Punto donde la coroutine *cede control* al event loop. No bloquea. |
| **Event Loop** | El "director de orquesta" que decide qué coroutine ejecutar. |
| **Task** | Coroutine envuelta para ser ejecutada concurrentemente. |

In [ ]:
import asyncio
import time


async def descargar_async(nombre: str, segundos: float) -> str:
    """Simula descarga asíncrona."""
    print(f"  [INICIO] Descargando {nombre}...")
    await asyncio.sleep(segundos)  # Cede control al event loop
    msg = f"  [LISTO] {nombre} descargado ({segundos}s)"
    print(msg)
    return msg


async def main_async() -> None:
    """Lanza todas las descargas de forma concurrente."""
    start = time.perf_counter()

    # asyncio.gather ejecuta todas las coroutines concurrentemente
    resultados = await asyncio.gather(
        descargar_async("archivo_A.dat", 0.5),
        descargar_async("archivo_B.dat", 0.5),
        descargar_async("archivo_C.dat", 0.5),
    )

    elapsed = time.perf_counter() - start
    print(f"\n  Total asyncio: {elapsed:.3f}s")
    print(f"  Secuencial hubiera sido: 1.500s")
    print(f"  Speedup: {1.5 / elapsed:.2f}x")


# Ejecutar el event loop
await main_async()

### ¿Cuándo preferir `asyncio` sobre `threading`?

| Criterio | `threading` | `asyncio` |
|----------|------------|-----------|
| **Miles de conexiones** | Alto overhead (1 hilo por conexión) | Bajo overhead (1 hilo, miles de tasks) |
| **Código existente** | Funciona con cualquier librería Python | Requiere librerías async (`aiohttp`, `aiofiles`) |
| **Complejidad** | Sincronización con Locks/Events | Coroutines y `await` (mentalidad diferente) |
| **Servidores web** | Funciona, pero no escala bien | Diseñado para esto (uvicorn, aiohttp) |

> **Nota**: El detalle completo de `asyncio` se cubre en **A02: Asyncio en Profundidad**.

---
## 7. Benchmarking: Midiendo Rendimiento

No puedes mejorar lo que no mides. Aprendamos a medir correctamente.

### Herramientas de medición

| Herramienta | Precisión | Uso |
|-------------|-----------|-----|
| `time.time()` | Baja (~15ms) | ❌ No recomendado para benchmarks |
| `time.perf_counter()` | **Alta** (~50ns) | ✅ Medir elapsed time |
| `time.process_time()` | Alta | ✅ Medir solo tiempo de CPU |
| `timeit` | Muy alta | ✅ Micro-benchmarks, promedia N ejecuciones |

In [ ]:
import time
import timeit

# Demo: las diferencias entre cada método de medición

def trabajo_mixto() -> None:
    """Mezcla CPU + I/O para ver la diferencia entre time methods."""
    # CPU work
    total = sum(i * i for i in range(1_000_000))
    # I/O wait
    time.sleep(0.1)


# 1. time.time() - No recomendado
t0 = time.time()
trabajo_mixto()
t1 = time.time()
print(f"time.time():         {t1 - t0:.6f}s")

# 2. time.perf_counter() - Recomendado para elapsed time
t0 = time.perf_counter()
trabajo_mixto()
t1 = time.perf_counter()
print(f"time.perf_counter(): {t1 - t0:.6f}s")

# 3. time.process_time() - Solo CPU time (excluye sleep/IO)
t0 = time.process_time()
trabajo_mixto()
t1 = time.process_time()
print(f"time.process_time(): {t1 - t0:.6f}s  ← Solo CPU, no incluye el sleep")

# 4. timeit - Para micro-benchmarks
t = timeit.timeit("sum(i*i for i in range(100_000))", number=10)
print(f"timeit (10 runs):   {t:.6f}s  ({t/10:.6f}s por run)")

### Benchmark completo: Threading vs Multiprocessing vs Asyncio

In [ ]:
import asyncio
import time
from multiprocessing import Pool
from threading import Thread

# --- Tarea I/O Bound ---
def io_task(_: int) -> float:
    time.sleep(0.2)
    return 0.2


async def io_task_async(_: int) -> float:
    await asyncio.sleep(0.2)
    return 0.2


# --- Tarea CPU Bound ---
def cpu_task(n: int) -> int:
    return sum(i * i for i in range(n))


NUM_TASKS = 6
CPU_ITERATIONS = 5_000_000

print("=" * 60)
print(f"BENCHMARK: {NUM_TASKS} tareas")
print("=" * 60)

# --- I/O BOUND ---
print("\n📦 I/O BOUND (sleep 0.2s x 6):")
print("-" * 40)

# Secuencial
start = time.perf_counter()
for i in range(NUM_TASKS):
    io_task(i)
t_seq = time.perf_counter() - start
print(f"  Secuencial:     {t_seq:.3f}s")

# Threading
start = time.perf_counter()
hilos = [Thread(target=io_task, args=(i,)) for i in range(NUM_TASKS)]
for t in hilos:
    t.start()
for t in hilos:
    t.join()
t_thr = time.perf_counter() - start
print(f"  Threading:      {t_thr:.3f}s ({t_seq/t_thr:.1f}x)")

# Asyncio
async def bench_async():
    return await asyncio.gather(*(io_task_async(i) for i in range(NUM_TASKS)))

start = time.perf_counter()
asyncio.run(bench_async())
t_async = time.perf_counter() - start
print(f"  Asyncio:        {t_async:.3f}s ({t_seq/t_async:.1f}x)")

# --- CPU BOUND ---
print(f"\n🔥 CPU BOUND (sumatoria {CPU_ITERATIONS:,} x 6):")
print("-" * 40)

# Secuencial
start = time.perf_counter()
for i in range(NUM_TASKS):
    cpu_task(CPU_ITERATIONS)
t_seq = time.perf_counter() - start
print(f"  Secuencial:     {t_seq:.3f}s")

# Threading
start = time.perf_counter()
hilos = [Thread(target=cpu_task, args=(CPU_ITERATIONS,)) for _ in range(NUM_TASKS)]
for t in hilos:
    t.start()
for t in hilos:
    t.join()
t_thr = time.perf_counter() - start
print(f"  Threading:      {t_thr:.3f}s ({t_seq/t_thr:.2f}x) ← GIL LO FRENÓ")

# Multiprocessing
start = time.perf_counter()
with Pool(NUM_TASKS) as pool:
    pool.map(cpu_task, [CPU_ITERATIONS] * NUM_TASKS)
t_mp = time.perf_counter() - start
print(f"  Multiprocessing:{t_mp:.3f}s ({t_seq/t_mp:.1f}x) ← ¡SÍ ACELERA!")

---
## Tabla Comparativa: Threading vs Multiprocessing vs Asyncio

| Criterio | `threading` | `multiprocessing` | `asyncio` |
|----------|------------|-------------------|----------|
**Tipo de concurrencia** | Preemptiva (OS scheduler) | Preemptiva (procesos separados) | Cooperativa (event loop) |
**Ideal para** | I/O bound (pocos hilos) | CPU bound (N cores) | I/O bound (miles de conexiones) |
**GIL impacto** | Bloquea CPU puro | Sin GIL (procesos separados) | Sin GIL (un solo hilo) |
**Overhead memoria** | Bajo (~8MB por hilo) | Alto (~50MB+ por proceso) | Muy bajo (~KB por task) |
**Comunicación** | Memoria compartida (Lock!) | IPC (pickle, Queue) | Memoria compartida (sin locks) |
**Escalabilidad** | ~100-1000 hilos | ~N cores (4-64) | ~100K+ tasks |
**Complejidad** | Baja-Media | Media | Media-Alta (cambio de mentalidad) |
**Librerías** | Cualquier librería | Cualquier librería | Requiere libs async |

### Flujo de decisión

```
¿Qué tipo de tarea es tu botella de agua?
│
├─ I/O Bound
│   ├─ < 100 conexiones → threading
│   └─ > 1000 conexiones → asyncio
│
├─ CPU Bound
│   ├─ 1 core → secuencial (optimiza algoritmo)
│   └─ N cores → multiprocessing
│
└─ Mixto (I/O + CPU)
    ├─ I/O en threads + CPU en Pool → Combinar
    └─ async + ProcessPoolExecutor → Combinar moderno
```

---
## Ejercicios

### Ejercicio 1: Clasificar tareas (Guiado)

Clasifica cada una de las siguientes funciones como **I/O bound**, **CPU bound**, o **Mixta**. Justifica tu respuesta.

In [ ]:
import hashlib
import time
from pathlib import Path


# Función A
def buscar_en_archivo(ruta: str, patron: str) -> list[str]:
    """Busca un patrón en un archivo de texto."""
    resultados = []
    with open(ruta, 'r', encoding='utf-8') as f:
        for linea in f:
            if patron in linea:
                resultados.append(linea.strip())
    return resultados


# Función B
def calcular_primos(limite: int) -> int:
    """Cuenta números primos hasta el límite."""
    def es_primo(n: int) -> bool:
        if n < 2:
            return False
        for i in range(2, int(n**0.5) + 1):
            if n % i == 0:
                return False
        return True

    return sum(1 for n in range(limite) if es_primo(n))


# Función C
def descargar_paginas(urls: list[str]) -> list[str]:
    """Descarga múltiples páginas web."""
    import urllib.request
    contenidos = []
    for url in urls:
        with urllib.request.urlopen(url, timeout=5) as resp:
            contenidos.append(resp.read().decode('utf-8'))
    return contenidos


print("EJERCICIO 1: Clasifica cada función:")
print("-" * 50)
print("A) buscar_en_archivo: ¿I/O, CPU, o Mixta?")
print("   Tu respuesta: ___")
print()
print("B) calcular_primos: ¿I/O, CPU, o Mixta?")
print("   Tu respuesta: ___")
print()
print("C) descargar_paginas: ¿I/O, CPU, o Mixta?")
print("   Tu respuesta: ___")
print()
print("Pista: ¿Qué dominó el tiempo? ¿La espera o el cálculo?")

<details>
<summary>💡 Respuestas (click para expandir)</summary>

- **A) I/O bound**: Leer disco es I/O. El CPU hace muy poco (solo comparar strings).
- **B) CPU bound**: Cálculos puros con loops. El GIL NO se libera.
- **C) I/O bound**: Esperar respuestas de red. El CPU está ocioso.

</details>

---

### Ejercicio 2: Race Condition (Guiado)

El siguiente código tiene un **race condition**. Identifícalo y corrígelo.

In [ ]:
import threading

# Cuenta bancaria compartida
saldo = 1000


def retirar(monto: int) -> None:
    """Retira dinero del saldo compartido."""
    global saldo
    # ¿Qué pasa si dos hilos llegan aquí al mismo tiempo?
    if saldo >= monto:
        time.sleep(0.001)  # Simula un pequeño delay (DB, procesamiento)
        saldo -= monto
        print(f"  Retirado ${monto}. Saldo: ${saldo}")
    else:
        print(f"  Fondos insuficientes para ${monto}. Saldo: ${saldo}")


import time

# Intentar retirar $800 dos veces con $1000 disponibles
saldo = 1000
print("Sin protección (puede dar saldo negativo):")
hilos = [
    threading.Thread(target=retirar, args=(800,)),
    threading.Thread(target=retirar, args=(800,)),
]
for t in hilos:
    t.start()
for t in hilos:
    t.join()
print(f"  Saldo final: ${saldo}")

# TU TAREA: Corregir usando threading.Lock()
print("\nTu tarea: Agrega un Lock para proteger saldo.")
print("Solución en la celda siguiente.")

In [ ]:
# SOLUCIÓN Ejercicio 2
import threading
import time

saldo = 1000
banco_lock = threading.Lock()


def retirar_seguro(monto: int) -> None:
    global saldo
    with banco_lock:  # Solo un hilo puede modificar saldo a la vez
        if saldo >= monto:
            time.sleep(0.001)
            saldo -= monto
            print(f"  Retirado ${monto}. Saldo: ${saldo}")
        else:
            print(f"  Fondos insuficientes para ${monto}. Saldo: ${saldo}")


saldo = 1000
print("Con Lock (saldo siempre >= 0):")
hilos = [
    threading.Thread(target=retirar_seguro, args=(800,)),
    threading.Thread(target=retirar_seguro, args=(800,)),
]
for t in hilos:
    t.start()
for t in hilos:
    t.join()
print(f"  Saldo final: ${saldo}")

---

### Ejercicio 3: Benchmark Comparativo (Guiado)

Mide el rendimiento de `threading`, `multiprocessing` y `asyncio` para ambas tareas (I/O y CPU).

In [ ]:
import time
import asyncio
from threading import Thread
from multiprocessing import Pool, Process
import os


def bench_io(n: int = 8, delay: float = 0.15) -> dict[str, float]:
    """Benchmark de I/O bound con las 3 estrategias."""
    results = {}

    def io_task(_: int) -> None:
        time.sleep(delay)

    async def io_task_async(_: int) -> None:
        await asyncio.sleep(delay)

    # Secuencial
    start = time.perf_counter()
    for i in range(n):
        io_task(i)
    results['secuencial'] = time.perf_counter() - start

    # Threading
    start = time.perf_counter()
    threads = [Thread(target=io_task, args=(i,)) for i in range(n)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    results['threading'] = time.perf_counter() - start

    # Asyncio
    async def run_async():
        await asyncio.gather(*(io_task_async(i) for i in range(n)))

    start = time.perf_counter()
    asyncio.run(run_async())
    results['asyncio'] = time.perf_counter() - start

    return results


def bench_cpu(n: int = 5_000_000, workers: int = 4) -> dict[str, float]:
    """Benchmark de CPU bound con las 3 estrategias."""
    results = {}

    def cpu_task(_: int) -> int:
        return sum(i * i for i in range(n))

    # Secuencial
    start = time.perf_counter()
    for i in range(workers):
        cpu_task(i)
    results['secuencial'] = time.perf_counter() - start

    # Threading
    start = time.perf_counter()
    threads = [Thread(target=cpu_task, args=(i,)) for i in range(workers)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    results['threading'] = time.perf_counter() - start

    # Multiprocessing
    start = time.perf_counter()
    with Pool(workers) as pool:
        pool.map(cpu_task, list(range(workers)))
    results['multiprocessing'] = time.perf_counter() - start

    return results


print("Benchmark I/O Bound (8 tareas, 0.15s cada una):")
print("-" * 50)
io_results = bench_io()
for name, t in io_results.items():
    speedup = io_results['secuencial'] / t
    print(f"  {name:15s}: {t:.3f}s ({speedup:.2f}x)")

print("\nBenchmark CPU Bound (5M iteraciones x 4 workers):")
print("-" * 50)
cpu_results = bench_cpu()
for name, t in cpu_results.items():
    speedup = cpu_results['secuencial'] / t

---

### Ejercicio 4 (Independiente): Pipeline de Datos

**Objetivo**: Crea un pipeline que combine las 3 técnicas.

**Escenario**:
1. **Descargar** 10 archivos simulados (I/O → usa `asyncio`)
2. **Procesar** cada archivo con cálculos pesados (CPU → usa `multiprocessing`)
3. **Guardar** resultados (I/O → usa `threading`)

**Requisitos**:
- Usa `asyncio` para la fase de descarga (simula con `asyncio.sleep`)
- Usa `multiprocessing.Pool` para la fase de procesamiento
- Usa `threading.Thread` para la fase de guardado
- Mide el tiempo total y de cada fase

**Estructura sugerida**:

```python
import asyncio
import time
from multiprocessing import Pool
from threading import Thread


# Fase 1: Descarga (asyncio)
async def descargar_simulado(id: int) -> dict:
    await asyncio.sleep(0.1)
    return {'id': id, 'datos': list(range(1000))}


# Fase 2: Procesamiento (multiprocessing)
def procesar(dato: dict) -> dict:
    resultado = sum(x**2 for x in dato['datos'])
    return {'id': dato['id'], 'resultado': resultado}


# Fase 3: Guardado (threading)
def guardar(resultado: dict) -> None:
    time.sleep(0.05)  # Simula escritura a disco
    print(f"  Guardado: id={resultado['id']}, resultado={resultado['resultado']}")


# Tu código aquí: orquesta las 3 fases
```

<details>
<summary>💡 Pistas</summary>

1. Para asyncio: `await asyncio.gather(*[descargar_simulado(i) for i in range(10)])`
2. Para multiprocessing: `Pool(4).map(procesar, descargas)`
3. Para threading: crea hilos para cada resultado y haz `join()` a todos
4. Para medir: usa `time.perf_counter()` al inicio y final de cada fase

</details>

In [ ]:
# Solución Ejercicio 4: Pipeline de datos
import asyncio
import time
from multiprocessing import Pool
from threading import Thread


# Fase 1: Descarga (asyncio)
async def descargar_simulado(id_: int) -> dict:
    """Simula descarga de archivo (I/O bound → asyncio)."""
    await asyncio.sleep(0.1)
    return {'id': id_, 'datos': list(range(1000))}


# Fase 2: Procesamiento (multiprocessing)
def procesar(dato: dict) -> dict:
    """Procesamiento CPU intensivo → multiprocessing."""
    resultado = sum(x**2 for x in dato['datos'])
    return {'id': dato['id'], 'resultado': resultado}


# Fase 3: Guardado (threading)
def guardar(resultado: dict) -> None:
    """Simula escritura a disco (I/O → threading)."""
    time.sleep(0.05)
    print(f"  Guardado: id={resultado['id']}, resultado={resultado['resultado']}")


# Orquestación del pipeline
def ejecutar_pipeline(num_archivos: int = 10) -> None:
    tiempos = {}

    # Fase 1: Descarga con asyncio
    start = time.perf_counter()
    descargas = asyncio.run(
        asyncio.gather(*(descargar_simulado(i) for i in range(num_archivos)))
    )
    tiempos['descarga'] = time.perf_counter() - start

    # Fase 2: Procesamiento con multiprocessing
    start = time.perf_counter()
    with Pool(4) as pool:
        resultados = pool.map(procesar, descargas)
    tiempos['procesamiento'] = time.perf_counter() - start

    # Fase 3: Guardado con threading
    start = time.perf_counter()
    hilos = [Thread(target=guardar, args=(r,)) for r in resultados]
    for t in hilos:
        t.start()
    for t in hilos:
        t.join()
    tiempos['guardado'] = time.perf_counter() - start

    # Reporte
    total = sum(tiempos.values())
    print(f"\n{'='*50}")
    print("PIPELINE COMPLETADO")
    print(f"{'='*50}")
    print(f"  Descarga (asyncio):    {tiempos['descarga']:.3f}s")
    print(f"  Procesamiento (MP):    {tiempos['procesamiento']:.3f}s")
    print(f"  Guardado (threading):  {tiempos['guardado']:.3f}s")
    print(f"  {'─'*40}")
    print(f"  TOTAL:                 {total:.3f}s")


ejecutar_pipeline()

---
## Resumen

### Conceptos clave

| Concepto | Definición |
|----------|------------|
**Concurrencia** | Múltiples tareas progresan *interleaving* (no necesariamente al mismo tiempo) |
**Paralelismo** | Múltiples tareas se ejecutan *simultáneamente* en distintos cores |
**I/O Bound** | El programa espera por red/disco/DB → solucionar con concurrencia |
**CPU Bound** | El programa calcula intensivamente → solucionar con paralelismo |
**GIL** | Mutex que impide ejecución paralela de Python puro en un solo proceso |

### Herramientas y cuándo usarlas

| Herramienta | Cuándo | Speedup esperado |
|------------|--------|------------------|
| `threading` | I/O bound, pocas conexiones | ~Nx (N = num tareas) |
| `multiprocessing` | CPU bound | ~Nx (N = num cores) |
| `asyncio` | I/O bound, miles de conexiones | ~Nx sin overhead de hilos |

### Buenas prácticas

1. **Mide primero**: Usa `time.perf_counter()`, no `time.time()`.
2. **Identifica el tipo**: ¿I/O o CPU? La respuesta determina la herramienta.
3. **No uses threads para CPU**: El GIL los serializa. Usa `multiprocessing`.
4. **Protege recursos compartidos**: Usa `threading.Lock()` cuando hilos modifican estado compartido.
5. **Considera asyncio**: Para servidores I/O-heavy con miles de conexiones.

### Siguiente paso

En **A02: Asyncio en Profundidad** profundizaremos en coroutines, el event loop, `async with/for`, `asyncio.Queue`, y patrones avanzados de concurrencia asíncrona.